# Crossmatch Ariel Targets vs MuSCAT-db Targets

Match targets from `ariel_targets.csv` and `muscatdb_targets.csv` by RA/Dec using `astropy.coordinates`.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

In [2]:
DATA = Path("../data")
SEPARATION_THRESHOLD = 2.0  # arcsec

In [3]:
ariel = pd.read_csv(DATA / "ariel_targets.csv")
muscat = pd.read_csv(DATA / "muscatdb_targets.csv", sep=",", on_bad_lines="skip", engine="python")
ariel.shape, muscat.shape

((1882, 107), (2095, 13))

In [4]:
# Drop rows with missing coordinates, then parse the MuSCAT-db export's
# sexagesimal coords (RA in hours, Dec in degrees) into decimal degrees.
# Malformed entries (e.g. "+36:36:561") are coerced to NaN and dropped.
from astropy.coordinates import Angle


def _to_deg(values, unit):
    out = []
    for v in values:
        try:
            out.append(Angle(str(v), unit=unit).deg)
        except Exception:
            out.append(np.nan)
    return np.array(out, dtype=float)


ariel_clean = ariel.dropna(subset=["Star RA", "Star Dec"]).reset_index(drop=True).copy()

muscat_clean = muscat.dropna(subset=["ra", "dec"]).copy()
muscat_clean["ra_deg"] = _to_deg(muscat_clean["ra"].values, u.hourangle)
muscat_clean["dec_deg"] = _to_deg(muscat_clean["dec"].values, u.deg)
muscat_clean = muscat_clean.dropna(subset=["ra_deg", "dec_deg"]).reset_index(drop=True)

ariel_clean.shape, muscat_clean.shape

((1882, 107), (1436, 15))

## Build SkyCoord objects and crossmatch

In [5]:
coord_ariel = SkyCoord(
    ra=ariel_clean["Star RA"].values * u.deg,
    dec=ariel_clean["Star Dec"].values * u.deg,
)
coord_muscat = SkyCoord(
    ra=muscat_clean["ra_deg"].values * u.deg,
    dec=muscat_clean["dec_deg"].values * u.deg,
)

In [6]:
# search_around_sky returns indices as (argument-coords, self-coords),
# so calling on coord_ariel with coord_muscat yields (muscat_idx, ariel_idx).
idx_muscat, idx_ariel, d2d, _ = coord_ariel.search_around_sky(
    coord_muscat, SEPARATION_THRESHOLD * u.arcsec
)
n_matches = len(idx_ariel)
n_ariel_total = len(ariel_clean)
n_muscat_total = len(muscat_clean)
n_matched_ariel = len(set(idx_ariel))
n_matched_muscat = len(set(idx_muscat))

print(f"Separation threshold: {SEPARATION_THRESHOLD}")
print(f"Ariel targets (with coords): {n_ariel_total}")
print(f"MuSCAT-db targets (with coords): {n_muscat_total}")
print(f"Ariel targets with a match: {n_matched_ariel}")
print(f"MuSCAT-db targets with a match: {n_matched_muscat}")
print(f"Total pair matches: {n_matches}")

Separation threshold: 2.0
Ariel targets (with coords): 1882
MuSCAT-db targets (with coords): 1436
Ariel targets with a match: 1
MuSCAT-db targets with a match: 1
Total pair matches: 1


## Inspect matched pairs

In [7]:
match_df = pd.DataFrame({
    "ariel_idx": idx_ariel,
    "muscat_idx": idx_muscat,
    "sep_arcsec": d2d.arcsec,
})
match_df = match_df.sort_values("sep_arcsec").reset_index(drop=True)

match_df["ariel_name"] = ariel_clean.iloc[idx_ariel]["Star Name"].values
match_df["ariel_tic"] = ariel_clean.iloc[idx_ariel]["TIC ID"].values
match_df["ariel_ra"] = ariel_clean.iloc[idx_ariel]["Star RA"].values
match_df["ariel_dec"] = ariel_clean.iloc[idx_ariel]["Star Dec"].values
match_df["muscat_name"] = muscat_clean.iloc[idx_muscat]["object"].values
match_df["muscat_ra"] = muscat_clean.iloc[idx_muscat]["ra_deg"].values
match_df["muscat_dec"] = muscat_clean.iloc[idx_muscat]["dec_deg"].values
match_df.head(10)

,ariel_idx,muscat_idx,sep_arcsec,ariel_name,ariel_tic,ariel_ra,ariel_dec,muscat_name,muscat_ra,muscat_dec
0,309,1224,0.069583,TOI-2887,134183038,121.985508,-39.014522,TOI2887,121.985525,-39.014536


In [8]:
match_df.muscat_name.unique()

array(['TOI2887'], dtype=object)

## Unmatched targets

In [9]:
matched_ariel_set = set(idx_ariel)
matched_muscat_set = set(idx_muscat)

unmatched_ariel = ariel_clean.iloc[
    [i for i in range(len(ariel_clean)) if i not in matched_ariel_set]
]
unmatched_muscat = muscat_clean.iloc[
    [i for i in range(len(muscat_clean)) if i not in matched_muscat_set]
]

print(f"Ariel targets NOT in MuSCAT-db: {len(unmatched_ariel)}")
print(f"MuSCAT-db targets NOT in Ariel: {len(unmatched_muscat)}")

Ariel targets NOT in MuSCAT-db: 1881
MuSCAT-db targets NOT in Ariel: 1435


In [10]:
unmatched_ariel[["Star Name", "TIC ID", "Star RA", "Star Dec"]].head()

,Star Name,TIC ID,Star RA,Star Dec
0,TOI-1007,65212867,112.752393,-4.463359
1,TOI-1009,107782586,111.667845,-24.462111
2,TOI-1019,341420329,120.147994,-54.878552
3,TOI-1027,20318757,167.133275,-29.653135
4,TOI-1027,20318757,167.133275,-29.653135


In [11]:
unmatched_muscat[["object", "ra", "dec"]].head()

,object,ra,dec
0,09881,08:47:34.72,+24:59:00.9
1,10302,16:23:58,+2:58:3
2,105140,21:52:19,+60:30:31
3,10mm,12:26:02.48,+34:35:53.0
4,136635,13:01:0,-21:54:27


## Export results

In [12]:
DATA / "crossmatch_results.csv"

PosixPath('../data/crossmatch_results.csv')

In [13]:
match_df.to_csv(DATA / "crossmatch_results.csv", index=False)
print("Saved to data/crossmatch_results.csv")

Saved to data/crossmatch_results.csv
